# 04 - XAI trên model tái hiện bài báo

Load đúng model đã train từ notebook 02 và chạy XAI trên model chính. SHAP/IG là phần giải thích chính; Grad-CAM chỉ được xem là bản điều chỉnh có giới hạn cho MLP dữ liệu bảng.

Source chính trên Drive: `/content/drive/MyDrive/Nhom28_CyberDetect_MLP_Final`. Không dùng folder Phase2/Phase3.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = '/content/drive/MyDrive/Nhom28_CyberDetect_MLP_Final'
DATA_CANDIDATES = [
    f'{PROJECT_PATH}/data/ton_iot.csv',
    f'{PROJECT_PATH}/data/raw/ton_iot.csv',
    '/content/ton_iot.csv',
]
DATA_PATH = next((p for p in DATA_CANDIDATES if os.path.exists(p)), DATA_CANDIDATES[0])
print('PROJECT_PATH =', PROJECT_PATH)
print('DATA_PATH =', DATA_PATH)
assert os.path.exists(PROJECT_PATH), f'Missing project folder: {PROJECT_PATH}'
assert os.path.exists(DATA_PATH), 'Missing ton_iot.csv inside project data folder'


In [ ]:
!pip -q install tensorflow pandas numpy scikit-learn matplotlib seaborn xgboost shap


In [ ]:
%cd {PROJECT_PATH}
!python - <<'PY'
import os, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, LabelEncoder

def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)
from sklearn.feature_selection import mutual_info_classif
import shap
PROJECT_PATH = '/content/drive/MyDrive/Nhom28_CyberDetect_MLP_Final'
DATA_PATH = os.environ.get('DATA_PATH', f'{PROJECT_PATH}/data/ton_iot.csv')
model_path = f'{PROJECT_PATH}/models/cyberdetect_mlp_paper_aligned.h5'
assert os.path.exists(model_path), 'Run notebook 02 first to create the paper-aligned model.'
os.makedirs(f'{PROJECT_PATH}/results/xai', exist_ok=True)
meta=json.load(open(f'{PROJECT_PATH}/models/paper_aligned_metadata.json', encoding='utf-8'))
df=pd.read_csv(DATA_PATH)
label_col=meta['label_column']; drop_cols=meta['dropped_columns']
X_raw=df.drop(columns=drop_cols, errors='ignore'); y_raw=df[label_col]
le=LabelEncoder(); y=le.fit_transform(y_raw)
Xtr,Xte,ytr,yte=train_test_split(X_raw,y,test_size=0.2,stratify=y,random_state=meta['seed'])
cat=Xtr.select_dtypes(include=['object','category','bool']).columns.tolist(); num=[c for c in Xtr.columns if c not in cat]
prep=ColumnTransformer([('num',Pipeline([('imputer',SimpleImputer(strategy='mean'))]),num),('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('encoder',make_one_hot_encoder())]),cat)], verbose_feature_names_out=False)
Xa=prep.fit_transform(Xtr).astype('float32'); Xb=prep.transform(Xte).astype('float32')
mi=mutual_info_classif(Xa,ytr,random_state=meta['seed']); idx=np.argsort(mi)[::-1][:30]
sc=MinMaxScaler(); Xtr2=sc.fit_transform(Xa[:,idx]); Xte2=sc.transform(Xb[:,idx])
features=meta['top_features']
model=load_model(model_path)
background=Xtr2[:100]; samples=Xte2[:50]
explainer=shap.GradientExplainer(model, background)
sv=explainer.shap_values(samples)
arr=sv[0] if isinstance(sv, list) else sv
if arr.ndim == 3: arr = arr[:,:,0]
shap.summary_plot(arr, pd.DataFrame(samples, columns=features), show=False)
plt.savefig(f'{PROJECT_PATH}/results/xai/shap_summary_paper_aligned.png', bbox_inches='tight', dpi=150); plt.close()
# Integrated gradients for one sample
x=tf.cast(samples[:1], tf.float32); baseline=tf.zeros_like(x); alphas=tf.reshape(tf.linspace(0.0,1.0,51),(-1,1))
interp=baseline+alphas*(x-baseline)
with tf.GradientTape() as tape:
    tape.watch(interp); pred=model(interp); target=tf.argmax(model(x)[0]); score=pred[:,target]
grads=tape.gradient(score, interp); ig=(x-baseline)*tf.reduce_mean(grads, axis=0)
imp=np.abs(ig.numpy().ravel()); top=np.argsort(imp)[::-1][:20]
plt.figure(figsize=(9,7)); plt.barh([features[i] for i in top][::-1], imp[top][::-1]); plt.tight_layout(); plt.savefig(f'{PROJECT_PATH}/results/xai/integrated_gradients_paper_aligned.png', dpi=150); plt.close()
print('[OK] Saved XAI figures in results/xai')
PY


In [ ]:
import os
for p in [f'{PROJECT_PATH}/results/xai/shap_summary_paper_aligned.png', f'{PROJECT_PATH}/results/xai/integrated_gradients_paper_aligned.png']:
    print('[OK]' if os.path.exists(p) else '[MISSING]', p)
